# 🍈 DurianVision — YOLOv11 Training (Google Colab)

Notebook ini akan melatih model **YOLOv11** untuk mendeteksi **10 varietas durian** menggunakan dataset dari Roboflow.

## Cara Pakai:
1. Pastikan runtime sudah pakai **GPU** → `Runtime` → `Change runtime type` → `T4 GPU`
2. Jalankan semua cell dari atas ke bawah
3. Upload dataset zip dari Roboflow saat diminta
4. Tunggu training selesai
5. Download file `best.pt` di akhir

---

## 1️⃣ Cek GPU & Install Dependencies

In [ ]:
#@title 1.1 Cek GPU tersedia
import torch

print("=" * 50)
print("  🖥️  GPU Check")
print("=" * 50)

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_mem / (1024**3)
    print(f"✅ GPU: {gpu_name}")
    print(f"✅ VRAM: {vram:.1f} GB")
    print(f"✅ CUDA: {torch.version.cuda}")
else:
    print("❌ GPU TIDAK TERSEDIA!")
    print("   Buka: Runtime → Change runtime type → T4 GPU")
    print("   Lalu restart notebook ini.")

In [ ]:
#@title 1.2 Install Ultralytics (YOLOv11)
!pip install -q ultralytics>=8.3.0

import ultralytics
ultralytics.checks()
print(f"\n✅ Ultralytics {ultralytics.__version__} terinstall")

---
## 2️⃣ Upload & Siapkan Dataset

Pilih **SALAH SATU** dari 2 opsi di bawah ini:

### Opsi A: Upload File ZIP dari Roboflow (Manual)

1. Buka project Anda di [Roboflow](https://app.roboflow.com)
2. Klik **Export Dataset** → Format: **YOLOv11** → **Download zip**
3. Jalankan cell di bawah dan upload file zip-nya

In [ ]:
#@title 2A. Upload ZIP dataset dari Roboflow
import os
import zipfile
from google.colab import files

# Buat direktori kerja
WORK_DIR = "/content/durian-training"
DATASET_DIR = os.path.join(WORK_DIR, "dataset")
os.makedirs(DATASET_DIR, exist_ok=True)

print("📂 Upload file ZIP dataset dari Roboflow...")
print("   (Format export: YOLOv11 atau YOLOv8)\n")

uploaded = files.upload()

for filename in uploaded.keys():
    zip_path = os.path.join(WORK_DIR, filename)

    # Pindahkan file yang diupload
    os.rename(filename, zip_path)

    # Ekstrak
    print(f"\n📦 Mengekstrak {filename}...")
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(DATASET_DIR)
    print(f"✅ Diekstrak ke {DATASET_DIR}")

# Tampilkan isi folder
print(f"\n📁 Isi dataset:")
for root, dirs, fls in os.walk(DATASET_DIR):
    level = root.replace(DATASET_DIR, '').count(os.sep)
    indent = '  ' * level
    print(f"{indent}📂 {os.path.basename(root)}/")
    if level < 2:  # Hanya tampilkan 2 level
        sub_indent = '  ' * (level + 1)
        for f in fls[:5]:  # Tampilkan max 5 file
            print(f"{sub_indent}📄 {f}")
        if len(fls) > 5:
            print(f"{sub_indent}... dan {len(fls) - 5} file lainnya")

### Opsi B: Download Otomatis via Roboflow API

Jika Anda punya **API key** dan **project URL** dari Roboflow, gunakan cell di bawah.

> Dapatkan API key di: Roboflow → Settings → API Keys

In [ ]:
#@title 2B. Download otomatis dari Roboflow API (opsional)

#@markdown Isi 3 field di bawah ini, lalu jalankan cell:
ROBOFLOW_API_KEY = ""  #@param {type:"string"}
ROBOFLOW_WORKSPACE = ""  #@param {type:"string"}
ROBOFLOW_PROJECT = ""  #@param {type:"string"}
ROBOFLOW_VERSION = 1  #@param {type:"integer"}

if ROBOFLOW_API_KEY and ROBOFLOW_WORKSPACE and ROBOFLOW_PROJECT:
    !pip install -q roboflow

    from roboflow import Roboflow

    WORK_DIR = "/content/durian-training"
    DATASET_DIR = os.path.join(WORK_DIR, "dataset")

    rf = Roboflow(api_key=ROBOFLOW_API_KEY)
    project = rf.workspace(ROBOFLOW_WORKSPACE).project(ROBOFLOW_PROJECT)
    version = project.version(ROBOFLOW_VERSION)
    ds = version.download("yolov11", location=DATASET_DIR)

    print(f"\n✅ Dataset didownload ke: {DATASET_DIR}")
else:
    print("⏭️  Dilewati — gunakan Opsi A (upload manual) di atas.")

---
## 3️⃣ Validasi & Perbaiki Dataset

In [ ]:
#@title 3.1 Validasi struktur dataset & perbaiki path
import yaml
import glob

WORK_DIR = "/content/durian-training"
DATASET_DIR = os.path.join(WORK_DIR, "dataset")

# ---- Cari data.yaml (bisa di root atau subfolder) ----
yaml_candidates = glob.glob(os.path.join(DATASET_DIR, "**", "data.yaml"), recursive=True)

if not yaml_candidates:
    print("❌ data.yaml tidak ditemukan!")
    print("   Pastikan dataset dari Roboflow sudah diupload dengan benar.")
else:
    DATA_YAML = yaml_candidates[0]
    # Jika data.yaml ada di subfolder, gunakan folder itu sebagai DATASET_DIR
    DATASET_DIR = os.path.dirname(DATA_YAML)
    print(f"✅ data.yaml ditemukan: {DATA_YAML}")
    print(f"   Dataset dir: {DATASET_DIR}")

    # Baca data.yaml
    with open(DATA_YAML, 'r') as f:
        data_config = yaml.safe_load(f)

    print(f"\n📋 Konfigurasi dataset:")
    print(f"   Jumlah kelas (nc): {data_config.get('nc', '?')}")
    print(f"   Nama kelas: {data_config.get('names', '?')}")

    # ---- Cari folder train/valid ----
    train_dir = None
    valid_dir = None

    for name in ['train', 'training']:
        candidate = os.path.join(DATASET_DIR, name)
        if os.path.isdir(candidate):
            train_dir = candidate
            break

    for name in ['valid', 'val', 'validation']:
        candidate = os.path.join(DATASET_DIR, name)
        if os.path.isdir(candidate):
            valid_dir = candidate
            break

    # Hitung gambar
    def count_images(folder):
        if not folder:
            return 0
        img_dir = os.path.join(folder, 'images') if os.path.isdir(os.path.join(folder, 'images')) else folder
        count = 0
        for ext in ['*.jpg', '*.jpeg', '*.png', '*.JPG', '*.JPEG', '*.PNG']:
            count += len(glob.glob(os.path.join(img_dir, ext)))
        return count

    train_count = count_images(train_dir)
    valid_count = count_images(valid_dir)

    print(f"\n📊 Jumlah gambar:")
    print(f"   Train: {train_count} gambar {'✅' if train_count > 0 else '❌'}")
    print(f"   Valid: {valid_count} gambar {'✅' if valid_count > 0 else '❌'}")

    # ---- Perbaiki path di data.yaml ----
    train_images = os.path.join(train_dir, 'images') if train_dir and os.path.isdir(os.path.join(train_dir, 'images')) else train_dir
    valid_images = os.path.join(valid_dir, 'images') if valid_dir and os.path.isdir(os.path.join(valid_dir, 'images')) else valid_dir

    corrected_config = data_config.copy()
    corrected_config['train'] = train_images
    corrected_config['val'] = valid_images

    # Cek test folder
    for name in ['test']:
        test_candidate = os.path.join(DATASET_DIR, name)
        if os.path.isdir(test_candidate):
            test_images = os.path.join(test_candidate, 'images') if os.path.isdir(os.path.join(test_candidate, 'images')) else test_candidate
            corrected_config['test'] = test_images
            test_count = count_images(test_candidate)
            print(f"   Test:  {test_count} gambar ✅")

    # Simpan data.yaml yang sudah diperbaiki
    CORRECTED_YAML = os.path.join(DATASET_DIR, 'data_corrected.yaml')
    with open(CORRECTED_YAML, 'w') as f:
        yaml.dump(corrected_config, f, default_flow_style=False, allow_unicode=True)

    print(f"\n✅ Path diperbaiki → {CORRECTED_YAML}")
    print(f"\n📄 Isi data_corrected.yaml:")
    with open(CORRECTED_YAML, 'r') as f:
        print(f.read())

---
## 4️⃣ Konfigurasi Training

Ubah parameter di bawah sesuai kebutuhan Anda.

In [ ]:
#@title 4.1 Pengaturan Training

#@markdown ### Model
#@markdown Pilih ukuran model (makin besar = makin akurat, tapi lebih lambat):
BASE_MODEL = "yolo11n.pt"  #@param ["yolo11n.pt", "yolo11s.pt", "yolo11m.pt", "yolo11l.pt", "yolo11x.pt"]

#@markdown ### Training Parameters
EPOCHS = 150  #@param {type:"slider", min:30, max:500, step:10}
IMAGE_SIZE = 640  #@param [320, 416, 640, 800] {type:"raw"}
BATCH_SIZE = 16  #@param [4, 8, 16, 32] {type:"raw"}
LEARNING_RATE = 0.01  #@param {type:"number"}
PATIENCE = 30  #@param {type:"slider", min:10, max:100, step:5}

print("⚡ Konfigurasi Training:")
print(f"   Model      : {BASE_MODEL}")
print(f"   Epochs     : {EPOCHS}")
print(f"   Image Size : {IMAGE_SIZE}")
print(f"   Batch Size : {BATCH_SIZE}")
print(f"   LR         : {LEARNING_RATE}")
print(f"   Patience   : {PATIENCE}")

# Rekomendasi otomatis berdasarkan GPU Colab
if torch.cuda.is_available():
    vram = torch.cuda.get_device_properties(0).total_mem / (1024**3)
    if vram < 8 and BATCH_SIZE > 8:
        print(f"\n⚠️  VRAM hanya {vram:.0f}GB. Batch size {BATCH_SIZE} mungkin terlalu besar.")
        print(f"   Disarankan: batch size 8")

---
## 5️⃣ Mulai Training! 🏋️

In [ ]:
#@title 5.1 🚀 Jalankan Training YOLOv11
from ultralytics import YOLO
from datetime import datetime

# Path dataset
data_yaml = os.path.join(DATASET_DIR, 'data_corrected.yaml')
if not os.path.exists(data_yaml):
    data_yaml = DATA_YAML
    print(f"⚠️  Menggunakan data.yaml original: {data_yaml}")

print("=" * 60)
print("  🍈 DurianVision — YOLOv11 Training")
print("=" * 60)
print(f"  Dataset : {data_yaml}")
print(f"  Model   : {BASE_MODEL}")
print(f"  Epochs  : {EPOCHS}")
print(f"  ImgSize : {IMAGE_SIZE}")
print(f"  Batch   : {BATCH_SIZE}")
print("=" * 60)

# Load model
print(f"\n📥 Memuat model dasar {BASE_MODEL}...")
model = YOLO(BASE_MODEL)

# Training
print("\n🏋️ Training dimulai...\n")
start_time = datetime.now()

results = model.train(
    data=data_yaml,
    epochs=EPOCHS,
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    device=0,
    patience=PATIENCE,
    lr0=LEARNING_RATE,
    project="/content/durian-training/runs",
    name="durian-yolov11",
    exist_ok=True,
    # Augmentasi data
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=10.0,
    translate=0.1,
    scale=0.5,
    fliplr=0.5,
    flipud=0.0,
    mosaic=1.0,
    mixup=0.1,
    # Optimisasi
    optimizer="auto",
    cos_lr=True,
    close_mosaic=10,
    # Output
    save=True,
    save_period=25,
    plots=True,
    verbose=True,
)

elapsed = datetime.now() - start_time
print(f"\n{'=' * 60}")
print(f"  ✅ Training selesai dalam {elapsed}")
print(f"{'=' * 60}")

---
## 6️⃣ Evaluasi Model

In [ ]:
#@title 6.1 📊 Validasi model terbaik
from ultralytics import YOLO

best_pt = "/content/durian-training/runs/durian-yolov11/weights/best.pt"

if os.path.exists(best_pt):
    model = YOLO(best_pt)

    data_yaml = os.path.join(DATASET_DIR, 'data_corrected.yaml')
    if not os.path.exists(data_yaml):
        data_yaml = DATA_YAML

    metrics = model.val(data=data_yaml)

    print(f"\n{'=' * 60}")
    print(f"  📊 Hasil Validasi Model Terbaik")
    print(f"{'=' * 60}")
    print(f"  mAP50     : {metrics.box.map50:.4f}")
    print(f"  mAP50-95  : {metrics.box.map:.4f}")
    print(f"  Precision : {metrics.box.mp:.4f}")
    print(f"  Recall    : {metrics.box.mr:.4f}")
    print(f"{'=' * 60}")

    # Interpretasi
    map50 = metrics.box.map50
    if map50 >= 0.9:
        print(f"\n  🏆 EXCELLENT! mAP50 = {map50:.1%}")
    elif map50 >= 0.75:
        print(f"\n  ✅ GOOD! mAP50 = {map50:.1%}")
    elif map50 >= 0.5:
        print(f"\n  ⚠️  FAIR — mAP50 = {map50:.1%}. Pertimbangkan tambah data atau epoch.")
    else:
        print(f"\n  ❌ LOW — mAP50 = {map50:.1%}. Dataset mungkin perlu diperbaiki.")
else:
    print("❌ best.pt tidak ditemukan. Pastikan training sudah selesai.")

In [ ]:
#@title 6.2 📈 Tampilkan grafik training
from IPython.display import Image, display

run_dir = "/content/durian-training/runs/durian-yolov11"

plots = [
    ("results.png", "📈 Training Results (Loss & mAP)"),
    ("confusion_matrix_normalized.png", "🔢 Confusion Matrix (Normalized)"),
    ("F1_curve.png", "📉 F1 Score Curve"),
    ("PR_curve.png", "📉 Precision-Recall Curve"),
    ("P_curve.png", "📉 Precision Curve"),
    ("R_curve.png", "📉 Recall Curve"),
]

for filename, title in plots:
    filepath = os.path.join(run_dir, filename)
    if os.path.exists(filepath):
        print(f"\n{title}")
        display(Image(filename=filepath, width=800))
    else:
        print(f"⚠️  {filename} tidak ditemukan")

In [ ]:
#@title 6.3 🖼️ Contoh prediksi pada gambar validasi
from IPython.display import Image, display

run_dir = "/content/durian-training/runs/durian-yolov11"

val_images = [
    "val_batch0_pred.jpg",
    "val_batch1_pred.jpg",
    "val_batch2_pred.jpg",
]

for img_name in val_images:
    img_path = os.path.join(run_dir, img_name)
    if os.path.exists(img_path):
        print(f"\n🖼️ {img_name}")
        display(Image(filename=img_path, width=800))

---
## 7️⃣ Test Prediksi (Opsional)

Upload gambar durian untuk test model yang sudah ditraining.

In [ ]:
#@title 7.1 🧪 Test prediksi dengan gambar custom
from ultralytics import YOLO
from google.colab import files
from IPython.display import Image, display
import cv2

best_pt = "/content/durian-training/runs/durian-yolov11/weights/best.pt"

if os.path.exists(best_pt):
    print("📷 Upload gambar durian untuk test prediksi...\n")
    uploaded = files.upload()

    model = YOLO(best_pt)

    for filename in uploaded.keys():
        print(f"\n🔍 Prediksi untuk: {filename}")

        results = model.predict(
            source=filename,
            conf=0.5,
            save=True,
            project="/content/durian-training/test_results",
            name="predict",
            exist_ok=True
        )

        # Tampilkan hasil
        for r in results:
            print(f"   Terdeteksi {len(r.boxes)} objek:")
            for box in r.boxes:
                cls_name = r.names[int(box.cls[0])]
                conf = float(box.conf[0])
                print(f"   • {cls_name}: {conf:.1%}")

        # Tampilkan gambar hasil
        result_img = f"/content/durian-training/test_results/predict/{filename}"
        if os.path.exists(result_img):
            display(Image(filename=result_img, width=600))
else:
    print("❌ Model belum ditraining. Jalankan cell training terlebih dahulu.")

---
## 8️⃣ Download Model `best.pt` 📥

Download file model yang sudah ditraining untuk digunakan di DurianVision.

In [ ]:
#@title 8.1 💾 Download best.pt
from google.colab import files
import shutil

best_pt = "/content/durian-training/runs/durian-yolov11/weights/best.pt"
last_pt = "/content/durian-training/runs/durian-yolov11/weights/last.pt"

if os.path.exists(best_pt):
    # Info ukuran file
    size_mb = os.path.getsize(best_pt) / (1024 * 1024)
    print(f"📦 best.pt — {size_mb:.1f} MB")
    print(f"\n⬇️  Downloading best.pt...")
    files.download(best_pt)
else:
    print("❌ best.pt tidak ditemukan!")
    print("   Pastikan training sudah selesai di Step 5.")

In [ ]:
#@title 8.2 💾 Download last.pt (backup, opsional)
from google.colab import files

last_pt = "/content/durian-training/runs/durian-yolov11/weights/last.pt"

if os.path.exists(last_pt):
    size_mb = os.path.getsize(last_pt) / (1024 * 1024)
    print(f"📦 last.pt — {size_mb:.1f} MB")
    print(f"\n⬇️  Downloading last.pt...")
    files.download(last_pt)
else:
    print("⚠️  last.pt tidak ditemukan.")

In [ ]:
#@title 8.3 💾 Download semua hasil training (ZIP)
import shutil
from google.colab import files

run_dir = "/content/durian-training/runs/durian-yolov11"

if os.path.exists(run_dir):
    print("📦 Membuat ZIP dari semua hasil training...")
    zip_path = shutil.make_archive(
        "/content/durian-yolov11-results",
        'zip',
        run_dir
    )
    size_mb = os.path.getsize(zip_path) / (1024 * 1024)
    print(f"📦 {zip_path} — {size_mb:.1f} MB")
    print(f"\n⬇️  Downloading...")
    files.download(zip_path)
else:
    print("❌ Folder training tidak ditemukan.")

---
## ✅ Selesai!

### Langkah selanjutnya:

1. **Simpan `best.pt`** ke project DurianVision Anda
2. Taruh di: `d:\GUI Duren\packages\inference\models\best.pt`
3. Backend akan otomatis menggunakan model ini untuk deteksi

### File yang didownload:
| File | Fungsi |
|------|--------|
| `best.pt` | Model terbaik (gunakan ini!) |
| `last.pt` | Model terakhir (backup) |
| `durian-yolov11-results.zip` | Semua grafik & hasil training |

---
*🍈 DurianVision — Deteksi Varietas Durian dengan AI*